# 03 — Logistic Growth Model (S-Curve) Fitting

Fits $P(t) = \frac{K}{1 + e^{-r(t - t_0)}}$ for each country.
Predicts market saturation and death year for traditional PBX.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.models.logistic_growth import (
    fit_all_countries, predict_years, plot_all_fits, summarize_results
)

In [ ]:
panel = pd.read_csv('data/processed/panel_data.csv')
print(f"Panel loaded: {panel.shape[0]} rows, {panel.shape[1]} cols")
print(f"Countries: {panel['country'].unique().tolist()}")

## 3.1 Fit S-Curve for All Countries

In [ ]:
results = fit_all_countries(
    panel,
    penetration_col='fixed_subs_value',
    death_threshold=0.05,
)
print(f"Fitted {len(results)} countries.")
summary = summarize_results(results)
summary

## 3.2 Visualize All Fits

In [ ]:
fig = plot_all_fits(results, panel, penetration_col='fixed_subs_value', n_cols=3)
plt.savefig('data/processed/logistic_fits_all.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.3 Results Analysis

In [ ]:
converged = summary[summary['converged']]
print(f"Converged: {len(converged)}/{len(summary)} countries\n")
print("=== Earliest predicted death years ===")
death = converged.dropna(subset=['death_year']).sort_values('death_year')
for _, row in death.head(5).iterrows():
    print(f"  {row['country'].upper()}: {row['death_year']} (R²={row['r_squared']})")
print("\n=== Latest predicted death years ===")
for _, row in death.tail(5).iterrows():
    print(f"  {row['country'].upper()}: {row['death_year']} (R²={row['r_squared']})")
print("\n=== Best fit (highest R²) ===")
best = converged.sort_values('r_squared', ascending=False).head(3)
for _, row in best.iterrows():
    print(f"  {row['country'].upper()}: R²={row['r_squared']}, death={row['death_year']}")

## 3.4 Market Phase Classification

In [ ]:
current_year = 2026
def classify_market(row):
    if not row['converged'] or pd.isna(row['death_year']):
        return 'Unknown'
    if row['death_year'] <= current_year:
        return 'Dead/Declining'
    elif row['death_year'] <= current_year + 10:
        return 'Near Death (≤10yr)'
    else:
        return 'Fading (>10yr)'

converged = converged.copy()
converged['market_phase'] = converged.apply(classify_market, axis=1)
phase_counts = converged['market_phase'].value_counts()
print("\nMarket Phase Distribution:")
for phase, count in phase_counts.items():
    print(f"  {phase}: {count}")
print("\nPer-country breakdown:")
print(converged[['country', 'K', 'r', 't0', 'death_year', 'r_squared', 'market_phase']].to_string(index=False))

In [ ]:
# Save results
converged.to_csv('data/processed/logistic_results.csv', index=False)
print("Results saved to data/processed/logistic_results.csv")